In [1]:
from ollama import Client

client = Client(host="http://192.168.0.22:11434/")

In [4]:
stream = client.chat(
    model="qwen3.5:latest",
    messages=[{"role": "user", "content": "What is 178543 * 23?"}],
    stream=True,
)

in_thinking = False
content = ""
thinking = ""
for chunk in stream:
    if chunk.message.thinking:
        if not in_thinking:
            in_thinking = True
            print("Thinking:\n", end="", flush=True)
        print(chunk.message.thinking, end="", flush=True)
        # accumulate the partial thinking
        thinking += chunk.message.thinking
    elif chunk.message.content:
        if in_thinking:
            in_thinking = False
            print("\n\nAnswer:\n", end="", flush=True)
        print(chunk.message.content, end="", flush=True)
        # accumulate the partial content
        content += chunk.message.content

    # append the accumulated fields to the messages for the next request
    new_messages = [{"role": "assistant", thinking: thinking, content: content}]

Thinking:
ThinkingThinking Process:

1.  **Analyze the Request:** The user is asking for the product of two numbers: 178543 and 23.

2.  **Identify the Operation:** Multiplication.

3.  **Perform the Calculation:**
    *   Multiply 178543 by 23.
    *   Break it down:
        *   $178543 \times 3 = ?$
        *   $178543 \times 20 = ?$
        *   Sum the results.

    *   Step 1: $178543 \times 3$
        *   $3 \times 3 = 9$
        *   $3 \times 4 = 12$ (write 2, carry 1)
        *   $3 \times 5 = 15 + 1 = 16$ (write 6, carry 1)
        *   $3 \times 8 = 24 + 1 = 25$ (write 5, carry 2)
        *   $3 \times 7 = 21 + 2 = 23$ (write 3, carry 2)
        *   $3 \times 1 = 3 + 2 = 5$
        *   Result: $535629$

    *   Step 2: $178543 \times 20$
        *   $178543 \times 2 = ?$
        *   $2 \times 3 = 6$
        *   $2 \times 4 = 8$
        *   $2 \times 5 = 10$ (write 0, carry 1)
        *   $2 \times 8 = 16 + 1 = 17$ (write 7, carry 1)
        *   $2 \times 7 = 14 + 1 = 15$ (write

In [ ]:
def get_temperature(city: str) -> str:
    """Get the current temperature for a city.

    Args:
      city: The name of the city

    Returns:
      The current temperature for the city
    """
    temperatures = {
        "New York": "22°C",
        "London": "15°C",
        "Tokyo": "18°C",
    }
    return temperatures.get(city, "Unknown")


messages = [{"role": "user", "content": "What is the temperature in New York?"}]

while True:
    stream = client.chat(model="qwen3.5:latest", messages=messages, stream=True, tools=[get_temperature], think=True)
    thinking = ""
    content = ""
    tool_calls = []

    done_thinking = False
    # accumulate the partial fields
    for chunk in stream:
        if chunk.message.thinking:
            thinking += chunk.message.thinking
            print(chunk.message.thinking, end="", flush=True)
        if chunk.message.content:
            if not done_thinking:
                done_thinking = True
                print("\n")
            content += chunk.message.content
            print(chunk.message.content, end="", flush=True)
        if chunk.message.tool_calls:
            tool_calls.extend(chunk.message.tool_calls)
            print(chunk.message.tool_calls)

    # append accumulated fields to the messages
    if thinking or content or tool_calls:
        messages.append({"role": "assistant", "thinking": thinking, "content": content, "tool_calls": tool_calls})

    if not tool_calls:
        break

    for call in tool_calls:
        if call.function.name == "get_temperature":
            result = get_temperature(**call.function.arguments)
        else:
            result = "Unknown tool"
        messages.append({"role": "tool", "tool_name": call.function.name, "content": result})

The user is asking about the temperature in New York. I have a tool called "get_temperature" that can get the current temperature for a city. I need to call this function with "New York" as the city parameter.[ToolCall(function=Function(name='get_temperature', arguments={'city': 'New York'}))]
The function returned 22°C as the temperature in New York. I should provide this information to the user in a helpful response.

The current temperature in New York is 22°C (71.6°F).